In [60]:
import tensorflow as tf

In [61]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()



In [62]:
input_width = 28
input_height = 28
input_channels = 1
input_pixels = 784

n_convolutional1 =32
stride_convolutional1 = 1

n_convolutional2 = 64
stride_convolutional2 = 1

convolution1_k = 5
convolution2_k = 5

max_pool1_k = 2
max_pool2_k = 2

n_hidden = 1024
n_out = 10 

input_size_to_hidden1 = (input_width // (max_pool1_k * max_pool2_k)) * (input_height // (max_pool1_k * max_pool2_k)) * n_convolutional2

In [63]:
weights = { "wc1": tf.Variable(tf.random.normal([convolution1_k, convolution1_k, input_channels, n_convolutional1])),
            "wc2": tf.Variable(tf.random.normal([convolution2_k, convolution2_k, n_convolutional1, n_convolutional2])),
            "wd1": tf.Variable(tf.random.normal([input_size_to_hidden1, n_hidden])),
            "wout": tf.Variable(tf.random.normal([n_hidden, n_out])) }

biases = { "bc1": tf.Variable(tf.random.normal([n_convolutional1])),
            "bc2": tf.Variable(tf.random.normal([n_convolutional2])),
            "bh": tf.Variable(tf.random.normal([n_hidden])),
            "bout": tf.Variable(tf.random.normal([n_out])) }

In [64]:
def cnn(x ,weights, biases ,rate):
    x = tf.reshape(x, shape=[-1, input_width, input_height, input_channels])
    
    conv1 = conv( x, weights['wc1'], biases['bc1'], stride_convolutional1)
    conv_pool1 = max_pool(conv1, max_pool1_k)


    conv2 = conv( conv_pool1, weights['wc2'], biases['bc2'], stride_convolutional2)
    conv_pool2 = max_pool(conv2, max_pool2_k)

    hidden_input = tf.reshape(conv_pool2, shape=[-1, input_size_to_hidden1])
    hidden_output_before_activation =tf.add(tf.matmul(hidden_input, weights['wd1']), biases['bh'])
    hidden_output_before_dropout = tf.nn.relu(hidden_output_before_activation)
    hidden_output = tf.nn.dropout(hidden_output_before_dropout, rate)

    output = tf.add(tf.matmul(hidden_output, weights['wout']), biases['bout'])
    return output



In [65]:
def conv(x, W, b , strides =1):
    conv = tf.nn.conv2d(x, W, strides=[1, strides, strides , 1], padding='SAME')
    conv = tf.nn.bias_add(conv, b)
    return tf.nn.relu(conv)

In [66]:
def max_pool(x, k=2):
    return tf.nn.max_pool2d(x, ksize=[1, k, k, 1], strides=[1, k, k, 1], padding='SAME')

In [67]:
x_train = tf.cast(x_train, tf.float32) / 255.0
x_test  = tf.cast(x_test,  tf.float32) / 255.0


In [68]:
cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=pred, labels=tf.one_hot(y_train, depth=n_out)))

In [69]:
optimizer = tf.optimizers.Adam(learning_rate=0.01)

def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        pred = cnn(x_batch, weights, biases, rate=0.2)
        cost = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(
            logits=pred, labels=tf.one_hot(y_batch, depth=n_out)
        ))
    trainable_vars = list(weights.values()) + list(biases.values())
    gradients = tape.gradient(cost, trainable_vars)
    optimizer.apply_gradients(zip(gradients, trainable_vars))
    return cost

In [70]:
batch_size = 128
n_epochs = 10

for epoch in range(n_epochs):
    for i in range(0, len(x_train), batch_size):
        x_batch = x_train[i : i + batch_size]
        y_batch = y_train[i : i + batch_size]
        loss = train_step(x_batch, y_batch)
    print(f"Epoch {epoch+1}, Loss: {loss.numpy():.4f}")

Epoch 1, Loss: 279.5711
Epoch 2, Loss: 118.0341
Epoch 3, Loss: 109.8663
Epoch 4, Loss: 111.2607
Epoch 5, Loss: 9.6838
Epoch 6, Loss: 12.3254
Epoch 7, Loss: 42.8892
Epoch 8, Loss: 2.9559
Epoch 9, Loss: 21.8963
Epoch 10, Loss: 0.0000


In [71]:
pridictions = tf.argmax(cnn(x_test, weights, biases,rate=0.0), axis=1)
correct_predictions = tf.equal(pridictions, y_test)
accuracy = tf.reduce_mean(tf.cast(correct_predictions, tf.float32))
print(f"Test Accuracy: {accuracy.numpy() * 100:.2f}%")

Test Accuracy: 97.86%
